# Tugas Akhir 05 (TA-05): Simulasi Antrian Sistem Bengkel (2 Montir)
**Mata Praktikum:** Pemodelan dan Simulasi  
**Modul 5:** Implementasi Simulasi Antrian Manual (Python)  
**Dosen Pengampu:** Mohammad Bayu Anggara, S.Kom., M.Kom.

### Deskripsi Sistem
Sebuah bengkel memiliki **2 montir**. Pemilik mobil datang dan mendaftar di resepsionis. Jika ada montir yang bebas, mobil langsung masuk untuk diperbaiki. Jika kedua montir sedang sibuk, pemilik mobil menunggu di ruang tunggu. Setelah perbaikan selesai, pemilik mobil melakukan pembayaran di kasir (dilayani resepsionis) dan keluar dari bengkel.

In [1]:
import numpy as np

def simulasi_bengkel(num_mobil=20, num_montir=2, mean_iat=12.0,
                     mean_daftar=3.0, mean_perbaikan=25.0, std_perbaikan=5.0,
                     mean_bayar=4.0, seed=42):
    """
    Simulasi Kejadian Diskrit (DES) Manual Sistem Bengkel dengan 2 Montir.
    Alur: Kedatangan -> Pendaftaran (Resepsionis) -> Cek Montir (Antri/Perbaikan)
          -> Pembayaran (Kasir/Resepsionis) -> Kepergian.
    """
    np.random.seed(seed)

    # 1. Inisialisasi Variabel Sistem
    clock = 0.0
    montir_sibuk = 0
    antrian_bengkel = []
    event_calendar = []

    # 2. Bangkitkan Waktu Antar-Kedatangan & Waktu Layanan (menit)
    iat = np.random.exponential(mean_iat, num_mobil - 1)
    waktu_daftar = np.maximum(1.0, np.random.normal(mean_daftar, 0.8, num_mobil))
    waktu_perbaikan = np.maximum(5.0, np.random.normal(mean_perbaikan, std_perbaikan, num_mobil))
    waktu_bayar = np.maximum(1.0, np.random.normal(mean_bayar, 1.0, num_mobil))

    # 3. Jadwalkan Kedatangan Semua Mobil
    waktu_kedatangan = {}
    t_arr = 0.0
    for i in range(1, num_mobil + 1):
        if i > 1:
            t_arr += iat[i - 2]
        waktu_kedatangan[i] = t_arr
        event_calendar.append([t_arr, "ARRIVAL", i])
    event_calendar.sort()

    # 4. Variabel Statistik
    waktu_masuk_antrian = {}
    total_waktu_tunggu = 0.0
    maks_waktu_tunggu = 0.0
    total_waktu_di_sistem = 0.0
    mobil_selesai = 0
    maks_antrian = 0

    print(f"=== SIMULASI SISTEM BENGKEL ({num_montir} Montir, {num_mobil} Mobil) ===")
    print(f"{'Clock':<9}{'Event':<18}{'Mobil':<8}{'Montir Sibuk':<14}{'Antrian':<10}{'Keterangan'}")
    print("-" * 82)

    # 5. Main Simulation Loop
    while event_calendar:
        clock, event_type, car_id = event_calendar.pop(0)

        if event_type == "ARRIVAL":
            # Mobil datang dan langsung mendaftar di resepsionis
            selesai_daftar = clock + waktu_daftar[car_id - 1]
            event_calendar.append([selesai_daftar, "SELESAI_DAFTAR", car_id])
            event_calendar.sort()
            print(f"{clock:<9.2f}{'ARRIVAL':<18}{car_id:<8}{montir_sibuk:<14}{len(antrian_bengkel):<10}Datang & mulai pendaftaran")

        elif event_type == "SELESAI_DAFTAR":
            # Cek ketersediaan montir (kapasitas 2 montir)
            if montir_sibuk < num_montir:
                montir_sibuk += 1
                selesai_perbaikan = clock + waktu_perbaikan[car_id - 1]
                event_calendar.append([selesai_perbaikan, "SELESAI_PERBAIKAN", car_id])
                event_calendar.sort()
                print(f"{clock:<9.2f}{'SELESAI_DAFTAR':<18}{car_id:<8}{montir_sibuk:<14}{len(antrian_bengkel):<10}Langsung diperbaiki montir")
            else:
                antrian_bengkel.append(car_id)
                waktu_masuk_antrian[car_id] = clock
                maks_antrian = max(maks_antrian, len(antrian_bengkel))
                print(f"{clock:<9.2f}{'SELESAI_DAFTAR':<18}{car_id:<8}{montir_sibuk:<14}{len(antrian_bengkel):<10}Montir penuh -> Ruang tunggu")

        elif event_type == "SELESAI_PERBAIKAN":
            # Perbaikan selesai -> proses pembayaran di kasir
            selesai_bayar = clock + waktu_bayar[car_id - 1]
            event_calendar.append([selesai_bayar, "DEPARTURE", car_id])

            # Cek antrian ruang tunggu untuk montir yang baru saja bebas
            if antrian_bengkel:
                next_car = antrian_bengkel.pop(0)
                tunggu = clock - waktu_masuk_antrian[next_car]
                total_waktu_tunggu += tunggu
                maks_waktu_tunggu = max(maks_waktu_tunggu, tunggu)
                selesai_next = clock + waktu_perbaikan[next_car - 1]
                event_calendar.append([selesai_next, "SELESAI_PERBAIKAN", next_car])
                ket = f"Ke kasir | Mobil {next_car} mulai diperbaiki"
            else:
                montir_sibuk -= 1
                ket = "Ke kasir | 1 Montir menjadi IDLE"

            event_calendar.sort()
            print(f"{clock:<9.2f}{'SELESAI_PERBAIKAN':<18}{car_id:<8}{montir_sibuk:<14}{len(antrian_bengkel):<10}{ket}")

        elif event_type == "DEPARTURE":
            # Pembayaran selesai dan mobil keluar dari bengkel
            mobil_selesai += 1
            total_waktu_di_sistem += (clock - waktu_kedatangan[car_id])
            print(f"{clock:<9.2f}{'DEPARTURE':<18}{car_id:<8}{montir_sibuk:<14}{len(antrian_bengkel):<10}Pembayaran selesai & keluar")

    # 6. Kalkulasi Statistik Akhir
    rata_tunggu = total_waktu_tunggu / mobil_selesai if mobil_selesai > 0 else 0.0
    rata_sistem = total_waktu_di_sistem / mobil_selesai if mobil_selesai > 0 else 0.0

    print("\n=== RINGKASAN HASIL SIMULASI BENGKEL ===")
    print(f"Total mobil dilayani         : {mobil_selesai} mobil")
    print(f"Rata-rata waktu tunggu       : {rata_tunggu:.2f} menit")
    print(f"Waktu tunggu maksimum        : {maks_waktu_tunggu:.2f} menit")
    print(f"Panjang antrian maksimum     : {maks_antrian} mobil")
    print(f"Rata-rata waktu di bengkel   : {rata_sistem:.2f} menit")
    print(f"Total waktu simulasi         : {clock:.2f} menit")

# Jalankan simulasi
simulasi_bengkel()


=== SIMULASI SISTEM BENGKEL (2 Montir, 20 Mobil) ===
Clock    Event             Mobil   Montir Sibuk  Antrian   Keterangan
----------------------------------------------------------------------------------
0.00     ARRIVAL           1       0             0         Datang & mulai pendaftaran
3.65     SELESAI_DAFTAR    1       1             0         Langsung diperbaiki montir
5.63     ARRIVAL           2       1             0         Datang & mulai pendaftaran
7.41     SELESAI_DAFTAR    2       2             0         Langsung diperbaiki montir
27.51    SELESAI_PERBAIKAN 1       1             0         Ke kasir | 1 Montir menjadi IDLE
31.30    DEPARTURE         1       1             0         Pembayaran selesai & keluar
34.36    SELESAI_PERBAIKAN 2       0             0         Ke kasir | 1 Montir menjadi IDLE
37.74    DEPARTURE         2       0             0         Pembayaran selesai & keluar
41.75    ARRIVAL           3       0             0         Datang & mulai pendaftaran
44.41 